In [1]:
import polars as pl
import altair as alt
import numpy as np

In [2]:
names = (
    pl.read_csv("figurelabels.csv", separator="&", has_header=False)
    .with_columns(pl.col("column_3").str.strip_chars_end(" \\"))
    .drop("column_2")
)
names

column_1,column_3
i64,str
1,"""-10.0"""
2,"""-10.0"""
3,"""-10.0"""
4,"""-10.0"""
5,"""10.0"""
…,…
46,"""0.0"""
47,"""0.0"""
48,"""0.0"""


In [3]:
min_val = pl.col("column_1").min()
max_val = pl.col("column_1").max()

range_string = (
    pl.when(min_val == max_val)
    .then(min_val.cast(pl.String))
    .otherwise(min_val.cast(pl.String) + "-" + max_val.cast(pl.String))
    .alias("time_range")  # Giving it a name to use in the next step
)

# The full pipeline
result = (
    names.filter(pl.col("column_3") != "0.0")
    .with_columns(pl.col("column_3").rle_id().alias("rle_id"))
    .group_by("rle_id")
    .agg(pl.col("column_3").first().cast(pl.Float32).alias("lag"), range_string)
    .group_by("lag")
    .agg(pl.col("time_range").str.join(", "))
)
for a, b in result.rows():
    print(b)
result

13-14, 16, 18, 20, 22, 24, 26
1-4, 11-12, 15, 17, 19, 21, 23, 25, 27-33, 36, 39-40
34-35, 37-38, 41-44
5-10


lag,time_range
f32,str
12.0,"""13-14, 16, 18, 20, 22, 24, 26"""
-10.0,"""1-4, 11-12, 15, 17, 19, 21, 23…"
1.25,"""34-35, 37-38, 41-44"""
10.0,"""5-10"""


In [4]:
df = pl.DataFrame(
    {
        "group_y": ("13-14, 16, 18, 20, 22, 24, 26",) * 48
        + ("5-10",) * 40
        + ("34-35, 37-38, 41-44",) * 5
        + ("1-4, 11-12, 15, 17, 19, 21, $$23, 25, 27-33, 36, 39-40",) * 41
        + ("target",) * 46,
        "group_c": ("13-14, 16, 18, 20, 22, 24, 26",) * 48
        + ("5-10",) * 40
        + ("34-35, 37-38, 41-44",) * 5
        + ("1-4, 11-12, 15, 17, 19, 21, $$23, 25, 27-33, 36, 39-40",) * 41
        + ("lag",) * 41
        + ("target",) * 5,
        "time": np.hstack(
            [
                np.arange(0.25, 12.01, 0.25),
                np.arange(0.25, 10.01, 0.25),
                np.arange(0.25, 1.26, 0.25),
                np.arange(-10, 0.01, 0.25),
                np.arange(-10, 1.26, 0.25),
            ]
        ),
    }
)
df

group_y,group_c,time
str,str,f64
"""13-14, 16, 18, 20, 22, 24, 26""","""13-14, 16, 18, 20, 22, 24, 26""",0.25
"""13-14, 16, 18, 20, 22, 24, 26""","""13-14, 16, 18, 20, 22, 24, 26""",0.5
"""13-14, 16, 18, 20, 22, 24, 26""","""13-14, 16, 18, 20, 22, 24, 26""",0.75
"""13-14, 16, 18, 20, 22, 24, 26""","""13-14, 16, 18, 20, 22, 24, 26""",1.0
"""13-14, 16, 18, 20, 22, 24, 26""","""13-14, 16, 18, 20, 22, 24, 26""",1.25
…,…,…
"""target""","""target""",0.25
"""target""","""target""",0.5
"""target""","""target""",0.75


In [5]:
base = alt.Chart(df).encode(
    x=alt.X("time:Q", axis=alt.Axis(title="Time", labelFontSize=16, titleFontSize=16, labelExpr="datum.label + 'h'", format="")),
    y=alt.Y(
        "group_y:N",
        sort=None,
        axis=alt.Axis(title="Feature Group", labelFontSize=14, titleFontSize=16, labelExpr="split(datum.value, '$$')"),
    ),
    color=alt.Color("group_c:N")
    .scale(
        domain=[
            "34-35, 37-38, 41-44",
            "13-14, 16, 18, 20, 22, 24, 26",
            "5-10",
            "1-4, 11-12, 15, 17, 19, 21, $$23, 25, 27-33, 36, 39-40",
            "lag",
            "target",
        ],  # Replace with your actual values
        range=["#1f77b4", "#1f77b4", "#1f77b4", "#1f77b4", "#1f77b4", "#d62728"],
    )
    .legend(None),
)
finished = (base.mark_point(size=50, filled=False) + base.mark_line(size=2)).properties(width=1000, height=300)
finished

alt.LayerChart(...)

In [6]:
finished.save("figures/timeline.png")